<h1 style='text-align:center; color:#2E4057'>🤖 Machine Learning from Scratch</h1>
<h3 style='text-align:center; color:#666'>Linear Regression · Logistic Regression · Decision Tree · Random Forest · XGBoost</h3>
<hr/>

## 📚 What You'll Learn

| # | Topic | Dataset Used |
|---|-------|-------------|
| 0 | Setup & Data Loading | — |
| 1 | **Linear Regression** — predict a number | 🏠 California Housing |
| 2 | **Logistic Regression** — predict a category | 🌸 Iris Flowers |
| 3 | **Decision Tree** — flowchart model | 🌸 Iris Flowers |
| 4 | **Random Forest** — many trees together | 🌸 Iris Flowers |
| 5 | **XGBoost** — the competition winner | 🌸 Iris Flowers |
| 6 | **Hyperparameter Tuning** — finding best settings | 🌸 Iris Flowers |
| 7 | **Feature Importance** — what matters most? | 🌸 Iris Flowers |
| 8 | **Evaluation Metrics** — measuring quality | Both |
| 9 | **Cross-Validation** — reliable testing | 🌸 Iris Flowers |
| 10 | **Model Cards** — document your model | — |

---

### 🌸 Why Iris & 🏠 California Housing?
These are the **"Hello World" datasets** of machine learning:
- **Iris**: 150 flower measurements → predict which of 3 flower species it is (classification)
- **California Housing**: Census data → predict house prices (regression)

Everyone in ML knows these datasets. They're simple, clean, and perfect for learning.

# 0. 🛠️ Setup — Installing & Importing Everything

**Think of this like unpacking your toolbox before starting work.**

We import:
- `pandas` — to work with tables of data (like Excel)
- `numpy` — for math operations
- `matplotlib` / `seaborn` — to draw charts
- `sklearn` — the main machine learning library
- `xgboost` — a powerful extra algorithm

In [1]:
# ── Core Libraries ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
%matplotlib inline

# ── Datasets built into sklearn ─────────────────────────────────
from sklearn.datasets import load_iris, fetch_california_housing

# ── Data Splitting & Preprocessing ──────────────────────────────
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler

# ── Models ──────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# ── Evaluation Metrics ──────────────────────────────────────────
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,   # regression
    accuracy_score, precision_score, recall_score,        # classification
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries loaded!')

✅ All libraries loaded!


## 0.1 Load Our Two Datasets

### 🌸 Dataset 1: Iris
- **150 flowers**, each with 4 measurements
- **Goal**: classify which of 3 species it belongs to

| Feature | What it measures |
|---------|------------------|
| sepal length (cm) | Length of the outer petal-like leaf |
| sepal width (cm) | Width of the outer petal-like leaf |
| petal length (cm) | Length of the inner colourful petal |
| petal width (cm) | Width of the inner colourful petal |
| **target** | 0=Setosa, 1=Versicolor, 2=Virginica |

### 🏠 Dataset 2: California Housing
- **20,640 census blocks** across California
- **Goal**: predict median house value (in $100,000s)

| Feature | What it means |
|---------|---------------|
| MedInc | Median income in block group |
| HouseAge | Median house age in block group |
| AveRooms | Average rooms per household |
| AveBedrms | Average bedrooms per household |
| Population | Block group population |
| AveOccup | Average household members |
| Latitude | Block group latitude |
| Longitude | Block group longitude |
| **target** | Median house value (×$100k) |

In [ ]:
# ── Load Iris (Classification) ───────────────────────────────────
iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['species'] = iris.target
iris_df['species_name'] = iris_df['species'].map(
    {0:'Setosa', 1:'Versicolor', 2:'Virginica'}
)
print('🌸 IRIS DATASET')
print(f'   Shape: {iris_df.shape} (rows × columns)')
print(f'   Classes: {iris_df.species_name.unique()}')
print()
iris_df.head(3)

In [ ]:
# ── Load California Housing (Regression) ─────────────────────────
housing = fetch_california_housing()
house_df = pd.DataFrame(housing.data, columns=housing.feature_names)
house_df['MedHouseVal'] = housing.target  # price in $100,000s
print('🏠 CALIFORNIA HOUSING DATASET')
print(f'   Shape: {house_df.shape} (rows × columns)')
print(f'   Price range: ${house_df.MedHouseVal.min()*100:.0f}k — ${house_df.MedHouseVal.max()*100:.0f}k')
print()
house_df.head(3)

In [ ]:
# ── Quick Explore: Iris ───────────────────────────────────────────
# Pairplot: see how features relate to each species
# Each dot = one flower. Color = species.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature distributions by species
for species, color in zip(['Setosa','Versicolor','Virginica'], ['#FF6B6B','#4ECDC4','#45B7D1']):
    subset = iris_df[iris_df.species_name == species]
    axes[0].scatter(subset['petal length (cm)'], subset['petal width (cm)'],
                    label=species, color=color, alpha=0.7, edgecolors='white', s=60)
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title('🌸 Iris: Petal Length vs Width\n(colours = species)')
axes[0].legend()

# Housing price distribution
axes[1].hist(house_df['MedHouseVal'], bins=50, color='#45B7D1', edgecolor='white')
axes[1].set_xlabel('Median House Value (×$100k)')
axes[1].set_ylabel('Count')
axes[1].set_title('🏠 California Housing:\nPrice Distribution')
axes[1].axvline(house_df['MedHouseVal'].mean(), color='red', linestyle='--', label='Mean')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Notice: Setosa is very easy to separate (far left cluster). Versicolor & Virginica overlap a bit.')

---
# 1. 📈 Linear Regression — Predicting a Number

## What is it? (Plain English)

**Linear Regression draws a straight line** through your data to predict numbers.

Imagine you're trying to predict **house price** based on **income**:
- High income area → expensive houses
- Low income area → cheaper houses
- A straight line through those points = your model!

The equation is literally from high school math:

$$\hat{y} = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$

where:
- $\hat{y}$ = your prediction (house price)
- $x_1, x_2...$ = your features (income, age, rooms...)
- $w_1, w_2...$ = **weights** the model learns ("how important is each feature?")
- $b$ = **bias/intercept** (the starting point)

## How does it learn?
It tries to **minimize the error** between predicted and actual prices. The error function is called **Mean Squared Error (MSE)**:

$$MSE = \frac{1}{n}\sum(y_{actual} - y_{predicted})^2$$

Think of it as: how far off are our guesses, on average?

## ⚠️ Important: Feature Scaling
Before linear/logistic regression, we must **standardize features** (make them all similar scale).
Why? If income is in thousands (50,000) and bedrooms are single digits (3), the model gets confused.
StandardScaler converts everything to: mean=0, std=1.

In [ ]:
# ── Step 1: Prepare Data ──────────────────────────────────────────
# X = features (what we know), y = target (what we want to predict)

X_house = house_df.drop('MedHouseVal', axis=1)   # all columns except price
y_house = house_df['MedHouseVal']                 # price is what we predict

# ── Step 2: Split into Train / Test ──────────────────────────────
# 80% of data for training the model
# 20% held back to TEST how well it does on new, unseen data
# random_state=42 makes the split reproducible (same result every run)

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)
print(f'Training samples: {X_h_train.shape[0]}')
print(f'Testing  samples: {X_h_test.shape[0]}')

# ── Step 3: Scale Features ────────────────────────────────────────
# RULE: Always fit the scaler on TRAINING data only!
# (Otherwise you 'leak' info from test into training — cheating!)

scaler = StandardScaler()
X_h_train_scaled = scaler.fit_transform(X_h_train)   # learn mean/std FROM train, then scale
X_h_test_scaled  = scaler.transform(X_h_test)        # only scale (use train's mean/std)

print(f'\nBefore scaling — MedInc range: {X_h_train["MedInc"].min():.1f} to {X_h_train["MedInc"].max():.1f}')
print(f'After scaling  — MedInc range: {X_h_train_scaled[:,0].min():.1f} to {X_h_train_scaled[:,0].max():.1f}')

In [ ]:
# ── Step 4: Train the Model ────────────────────────────────────────
# .fit() = 'learn from training data'
# Internally: finds the best weights w1, w2... to minimize MSE

lr_model = LinearRegression()
lr_model.fit(X_h_train_scaled, y_h_train)

print('✅ Model trained!')
print('\nWhat the model learned (weights / coefficients):')
coef_df = pd.DataFrame({
    'Feature': housing.feature_names,
    'Weight':  lr_model.coef_.round(4)
}).sort_values('Weight', key=abs, ascending=False)
print(coef_df.to_string(index=False))
print(f'\nBias (intercept): {lr_model.intercept_:.4f}')
print('\nInterpretation:')
print('  Positive weight → feature pushes price UP')
print('  Negative weight → feature pushes price DOWN')
print('  Larger |weight| → feature has MORE influence')

In [ ]:
# ── Step 5: Predict & Evaluate ────────────────────────────────────
y_h_pred = lr_model.predict(X_h_test_scaled)

# Regression Metrics:
mae  = mean_absolute_error(y_h_test, y_h_pred)
mse  = mean_squared_error(y_h_test, y_h_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_h_test, y_h_pred)

print('📊 LINEAR REGRESSION — Results on Test Set')
print('='*50)
print(f'  MAE  (Mean Absolute Error) : {mae:.4f}  ← avg error in $100k units')
print(f'       = off by ~${mae*100000:.0f} on average')
print(f'  RMSE (Root Mean Sq Error)  : {rmse:.4f}  ← penalizes big mistakes more')
print(f'  R²   (R-squared)           : {r2:.4f}  ← {r2*100:.1f}% of price variance explained')
print()
print('What is R²?')
print('  R² = 1.0 → perfect model (predicts everything correctly)')
print('  R² = 0.0 → model is no better than always predicting the mean')
print('  R² < 0.0 → model is WORSE than predicting the mean (very bad!)')

In [ ]:
# ── Visualize: Predicted vs Actual ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted scatter
# A PERFECT model would have all points on the red diagonal line
axes[0].scatter(y_h_test, y_h_pred, alpha=0.3, color='steelblue', s=10)
axes[0].plot([y_h_test.min(), y_h_test.max()],
             [y_h_test.min(), y_h_test.max()], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (×$100k)')
axes[0].set_ylabel('Predicted Price (×$100k)')
axes[0].set_title(f'Linear Regression: Predicted vs Actual\nR² = {r2:.3f}')
axes[0].legend()

# Plot 2: Residuals (errors)
# Good model: residuals scattered randomly around 0 (no pattern)
residuals = y_h_test - y_h_pred
axes[1].scatter(y_h_pred, residuals, alpha=0.3, color='coral', s=10)
axes[1].axhline(0, color='black', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('Residual Plot\n(should be random around 0 for a good model)')

plt.tight_layout()
plt.show()
print('💡 Notice: Linear regression struggles with very expensive houses (capped at $500k in this dataset)')

---
# 2. 🔵 Logistic Regression — Predicting a Category

## Wait — it's called 'regression' but it classifies?

Yes! Confusing name, but:
- **Linear Regression** → predicts a number (price, temperature, age)
- **Logistic Regression** → predicts a **probability**, then turns that into a class

## How it works (Baby Steps):

1. Start like Linear Regression: compute $z = w_1 x_1 + w_2 x_2 + ... + b$
2. Squeeze $z$ through the **Sigmoid function**: $\sigma(z) = \frac{1}{1+e^{-z}}$
3. Sigmoid always outputs a value between 0 and 1 → that's our **probability**
4. If probability > 0.5 → predict class 1, else class 0

```
Any number z  →  Sigmoid  →  0.0 to 1.0  →  Class 0 or 1
     z=0      →    0.5    →   50/50
     z=5      →    0.99   →   very likely Class 1
     z=-5     →    0.007  →   very likely Class 0
```

## Multiclass (3 flowers, not just 2)
For 3 classes, it uses **One-vs-Rest (OvR)**: trains 3 separate classifiers:
- Is it Setosa vs NOT Setosa?
- Is it Versicolor vs NOT Versicolor?  
- Is it Virginica vs NOT Virginica?

Picks whichever class has the highest probability.

In [ ]:
# ── Prepare Iris Data ─────────────────────────────────────────────
X_iris = iris_df[iris.feature_names]
y_iris = iris_df['species']

# Train/Test split — stratify=y means each class is equally represented in both splits
X_i_train, X_i_test, y_i_train, y_i_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Scale features for logistic regression
scaler_iris = StandardScaler()
X_i_train_sc = scaler_iris.fit_transform(X_i_train)
X_i_test_sc  = scaler_iris.transform(X_i_test)

print(f'Train: {X_i_train.shape[0]} samples | Test: {X_i_test.shape[0]} samples')
print('Class distribution (train):',
      dict(y_i_train.value_counts().rename({0:'Setosa', 1:'Versicolor', 2:'Virginica'})))

In [ ]:
# ── Train Logistic Regression ─────────────────────────────────────
log_reg = LogisticRegression(max_iter=200, random_state=42)
log_reg.fit(X_i_train_sc, y_i_train)

log_preds = log_reg.predict(X_i_test_sc)
log_proba = log_reg.predict_proba(X_i_test_sc)  # probabilities for each class

print('✅ Logistic Regression trained!')
print(f'Accuracy: {accuracy_score(y_i_test, log_preds):.4f}')
print()

# Show what the model outputs for a few examples
print('Example predictions (first 5 test samples):')
species_names = ['Setosa', 'Versicolor', 'Virginica']
for i in range(5):
    actual = species_names[y_i_test.iloc[i]]
    predicted = species_names[log_preds[i]]
    probs = log_proba[i]
    correct = '✅' if actual == predicted else '❌'
    print(f'  {correct} Actual: {actual:12s} | Predicted: {predicted:12s}'
          f' | Proba: [S={probs[0]:.2f} V={probs[1]:.2f} Vi={probs[2]:.2f}]')

In [ ]:
# ── Visualize the Sigmoid Function ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid curve
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

axes[0].plot(z, sigmoid, color='steelblue', lw=3)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary (0.5)')
axes[0].axvline(0,   color='gray', linestyle='--', alpha=0.5)
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.2, color='green', label='Predict Class 1')
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.2, color='red', label='Predict Class 0')
axes[0].set_xlabel('z  (linear combination of features)')
axes[0].set_ylabel('Probability')
axes[0].set_title('The Sigmoid Function\n(squeezes any number to 0–1 range)')
axes[0].legend()
axes[0].grid(True)

# Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_i_test, log_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Setosa','Versicolor','Virginica'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Logistic Regression\nConfusion Matrix')

plt.tight_layout()
plt.show()

print(classification_report(y_i_test, log_preds, target_names=['Setosa','Versicolor','Virginica']))

---
# 3. 🌳 Decision Tree — The Flowchart Model

## What is it? (Plain English)

A Decision Tree is literally a flowchart of yes/no questions.

Imagine a doctor diagnosing a flower:
```
Is petal length < 2.5cm?
├── YES → It's SETOSA! (done)
└── NO  → Is petal width < 1.8cm?
          ├── YES → It's probably VERSICOLOR
          └── NO  → It's probably VIRGINICA
```

## Key Terms:
| Term | Simple Meaning |
|------|----------------|
| **Root Node** | The very first question (top of tree) |
| **Branch** | The path you take after each yes/no answer |
| **Leaf Node** | The final answer (prediction) |
| **Depth** | How many levels deep the tree is |
| **Gini Impurity** | How mixed up the classes are at a node (0 = pure, 0.5 = 50/50 mixed) |

## The Overfitting Problem:
- **Shallow tree** (depth 2) → Too simple, misses patterns = **Underfitting**
- **Deep tree** (depth 20) → Memorizes training data, fails on new data = **Overfitting**
- **Just right** → Generalizes well to new data

In [ ]:
# ── Train Decision Tree ───────────────────────────────────────────
# Note: Decision Trees do NOT need scaled data (they use thresholds, not distances)

dt = DecisionTreeClassifier(
    max_depth=4,           # tree can ask at most 4 questions deep
    min_samples_split=5,   # need at least 5 samples to make a split
    random_state=42
)
dt.fit(X_i_train, y_i_train)  # use unscaled data — fine for trees!

dt_preds = dt.predict(X_i_test)
print(f'Decision Tree Accuracy: {accuracy_score(y_i_test, dt_preds):.4f}')
print(f'Tree depth actually used: {dt.get_depth()}')
print(f'Number of leaf nodes: {dt.get_n_leaves()}')

In [ ]:
# ── Visualize the Decision Tree ────────────────────────────────────
# This is one of the BEST things about Decision Trees: you can SEE exactly how they think!

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=['Setosa', 'Versicolor', 'Virginica'],
    filled=True,      # color nodes by majority class
    rounded=True,
    fontsize=11,
    ax=ax
)
plt.title('🌳 Decision Tree — Iris Classification\n'
          'Blue=Setosa  Orange=Versicolor  Green=Virginica', fontsize=15)
plt.tight_layout()
plt.show()

print('How to read each box:')
print('  Line 1: Feature ≤ threshold (the question being asked)')
print('  gini:   how impure/mixed is this node? (0=pure, 0.5=50-50)')
print('  samples: how many training samples reached this node')
print('  value:  [# Setosa, # Versicolor, # Virginica] at this node')
print('  class:  the predicted class (majority)')

In [ ]:
# ── Overfitting Demo: Tree Depth vs Accuracy ──────────────────────
# IMPORTANT CONCEPT: watch train vs test accuracy as depth increases

train_accs, test_accs = [], []
depths = range(1, 16)

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_i_train, y_i_train)
    train_accs.append(accuracy_score(y_i_train, clf.predict(X_i_train)))
    test_accs.append(accuracy_score(y_i_test,  clf.predict(X_i_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'o-', label='Train Accuracy', color='steelblue', lw=2)
plt.plot(depths, test_accs,  's-', label='Test Accuracy',  color='coral', lw=2)
plt.axvline(x=4, linestyle='--', color='gray', alpha=0.7, label='Our chosen depth=4')
plt.fill_between(depths, train_accs, test_accs, alpha=0.1, color='red', label='Overfitting gap')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Demonstration\nTrain vs Test Accuracy by Tree Depth')
plt.legend()
plt.xticks(depths)
plt.tight_layout()
plt.show()

print('💡 Key Insight:')
print('  Depth 1–3: Train ≈ Test  →  Underfitting (model too simple)')
print('  Depth ~4:  Both high     →  Sweet spot!')
print('  Depth 8+:  Train=100%, Test drops  →  Overfitting (memorizing training data)')

---
# 4. 🌲🌲🌲 Random Forest — Wisdom of the Crowd

## What is it? (Plain English)

Instead of relying on ONE decision tree, **build hundreds of trees and let them vote!**

It's like asking 100 different doctors for a diagnosis instead of just one. Even if each doctor makes small mistakes, they make **different** mistakes, and the majority vote is usually right.

## How it works (Step by Step):

**Step 1 — Bootstrap Sampling:**
For each of the 100 trees, randomly sample N rows from the training data **with replacement**.
(Some rows appear multiple times, some don't appear at all. This is called a "bootstrap" sample.)

**Step 2 — Random Feature Selection:**
At each split in each tree, only consider a **random subset** of features.
(For Iris: 4 features, so each split considers √4 = 2 random features.)
This makes trees different from each other!

**Step 3 — Vote:**
Each tree predicts a class. Final answer = whatever class gets the most votes.

```
Tree 1: Versicolor
Tree 2: Versicolor  →  VOTE  →  Versicolor wins!
Tree 3: Virginica
```

## Why it's better than one tree:
- Individual trees overfit, but their errors are **random and different**
- Averaging random errors → they cancel out!
- Result: **less variance, better generalization**

This technique is called **Bagging** (Bootstrap AGGregating)

In [ ]:
# ── Train Random Forest ───────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100,    # how many trees to build
    max_depth=6,         # max depth for each tree
    max_features='sqrt', # features per split = sqrt(total features)
    random_state=42,
    n_jobs=-1            # use all CPU cores (faster training)
)
rf.fit(X_i_train, y_i_train)

rf_preds = rf.predict(X_i_test)
print(f'Random Forest Accuracy: {accuracy_score(y_i_test, rf_preds):.4f}')

In [ ]:
# ── How many trees do we need? ────────────────────────────────────
# Let's see how accuracy changes as we add more trees

n_trees_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
rf_test_accs = []

for n in n_trees_range:
    clf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    clf.fit(X_i_train, y_i_train)
    rf_test_accs.append(accuracy_score(y_i_test, clf.predict(X_i_test)))

plt.figure(figsize=(10, 5))
plt.plot(n_trees_range, rf_test_accs, 'o-', color='forestgreen', lw=2, ms=8)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: How Many Trees Do We Need?')
plt.xticks(n_trees_range)
plt.ylim(0.85, 1.01)
plt.axhline(rf_test_accs[-1], linestyle='--', color='gray', alpha=0.5, label='Max accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print('💡 Key Insight: Performance stabilizes around 50-100 trees.')
print('   Adding more trees → more computation, barely any accuracy gain.')

---
# 5. ⚡ XGBoost — The Competition Winner

## What is it? (Plain English)

XGBoost also uses many decision trees, but instead of building them **in parallel** (Random Forest), it builds them **one at a time, each fixing the mistakes of the previous one**.

This technique is called **Boosting**.

## Visual Analogy:
Imagine a student who:
1. Takes a test, gets some wrong
2. Studies ONLY the questions they got wrong
3. Takes the test again, fewer mistakes
4. Studies the remaining wrong ones
5. Repeats until very few mistakes

## Bagging vs Boosting:
| | Random Forest (Bagging) | XGBoost (Boosting) |
|---|---|---|
| Trees built | All at once (parallel) | One by one (sequential) |
| Each tree trained on | Random sample of data | The ERRORS of previous trees |
| Aggregation | Majority vote | Weighted sum |
| Main fix | Reduces variance | Reduces bias |
| Speed | Faster (parallel) | Slower (sequential) |
| Often better when | Data is noisy | Data is clean |

## Key XGBoost Parameters:
- `n_estimators`: how many rounds of boosting (how many trees to add)
- `learning_rate` (eta): how much each new tree contributes. Small (0.01) = slow but careful. Large (0.5) = fast but may overshoot
- `max_depth`: depth of each individual tree
- `subsample`: fraction of rows used per tree (like Random Forest's bootstrap)
- `colsample_bytree`: fraction of features used per tree

In [ ]:
# ── Train XGBoost ─────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,      # each tree contributes 10% of its prediction
    subsample=0.8,          # use 80% of data per tree
    colsample_bytree=0.8,   # use 80% of features per tree
    eval_metric='mlogloss', # metric to monitor during training
    random_state=42,
    verbosity=0
)
xgb_model.fit(
    X_i_train, y_i_train,
    eval_set=[(X_i_test, y_i_test)],  # monitor test performance during training
    verbose=False
)

xgb_preds = xgb_model.predict(X_i_test)
print(f'XGBoost Accuracy: {accuracy_score(y_i_test, xgb_preds):.4f}')

In [ ]:
# ── XGBoost Learning Curve ────────────────────────────────────────
# Watch how the model improves with each boosting round

results = xgb_model.evals_result()

plt.figure(figsize=(10, 5))
plt.plot(results['validation_0']['mlogloss'], color='darkorange', lw=2, label='Test Log Loss')
plt.xlabel('Boosting Round (# of trees added)')
plt.ylabel('Log Loss (lower = better)')
plt.title('XGBoost Learning Curve\n(Model improves as more trees are added)')
plt.legend()
plt.tight_layout()
plt.show()

print('💡 What is Log Loss?')
print('   It measures how CONFIDENT and CORRECT the predictions are.')
print('   Being confident AND wrong is penalized heavily.')
print('   Lower log loss = better model.')
print('   Notice: it keeps improving as we add more trees!')

---
# 6. 🔧 Hyperparameter Tuning — Finding the Best Settings

## What are hyperparameters?

**Parameters** = values the model learns from data (e.g., tree split thresholds, weights)

**Hyperparameters** = settings YOU choose BEFORE training (e.g., max_depth, n_estimators, learning_rate)

It's like the knobs on an oven:
- The recipe learns from ingredients (parameters)
- But you set temperature & time before baking (hyperparameters)

## Two Search Strategies:

### Grid Search (Exhaustive)
Try EVERY single combination in your list.
- You specify: max_depth=[2,4,6] and n_estimators=[50,100]
- It tries: (2,50), (2,100), (4,50), (4,100), (6,50), (6,100) — all 6 combinations
- ✅ Guaranteed to find the best in your grid
- ❌ Slow if your grid is large

### Random Search (Sampling)
Try N random combinations from a distribution.
- You specify ranges: max_depth between 2-10, n_estimators between 50-500
- It randomly tries 20 combinations (or however many you say)
- ✅ Much faster for large search spaces
- ❌ Might miss the true best setting

## ⚠️ ALWAYS use Cross-Validation with tuning!
Without CV, you'd tune specifically for your test set → overfitting to the test set.

In [ ]:
# ── Grid Search on Decision Tree ─────────────────────────────────

param_grid = {
    'max_depth': [2, 3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}
# Total combinations: 5 × 3 × 2 = 30
# Each tested with 5-fold CV = 150 model fits total

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,               # 5-fold cross-validation
    scoring='accuracy', # optimize for accuracy
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_i_train, y_i_train)

print('🔍 Grid Search Results:')
print(f'  Best Parameters: {grid_search.best_params_}')
print(f'  Best CV Accuracy: {grid_search.best_score_:.4f}')
print(f'\n  This means: using these settings, across 5 different')
print(f'  train/test splits, the average accuracy was {grid_search.best_score_*100:.1f}%')

In [ ]:
# ── Visualize Grid Search Results ────────────────────────────────
results_df = pd.DataFrame(grid_search.cv_results_)
results_df['max_depth']        = results_df['params'].apply(lambda x: x['max_depth'])
results_df['min_samples_split']= results_df['params'].apply(lambda x: x['min_samples_split'])
results_df['criterion']        = results_df['params'].apply(lambda x: x['criterion'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, crit in zip(axes, ['gini', 'entropy']):
    pivot = results_df[results_df['criterion']==crit].pivot(
        index='max_depth', columns='min_samples_split', values='mean_test_score'
    )
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax,
                linewidths=0.5, vmin=0.9, vmax=1.0)
    ax.set_title(f'Grid Search CV Accuracy\n(criterion={crit})')

plt.suptitle('GridSearchCV: Mean 5-Fold Accuracy for Each Combo', fontsize=13)
plt.tight_layout()
plt.show()
print('💡 Darker blue = higher accuracy = better hyperparameter combo')

In [ ]:
# ── Random Search on Random Forest ───────────────────────────────
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 300),    # random integer from 50 to 299
    'max_depth':    randint(3, 10),
    'min_samples_split': randint(2, 20),
    'max_features': ['sqrt', 'log2']
}

rand_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,            # try 20 random combos (vs 30+ for exhaustive)
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rand_search.fit(X_i_train, y_i_train)

print('🎲 Randomized Search Results:')
print(f'  Best Parameters: {rand_search.best_params_}')
print(f'  Best CV Accuracy: {rand_search.best_score_:.4f}')

best_rf = rand_search.best_estimator_  # save for use later

---
# 7. 📊 Feature Importance — What Does the Model Care About?

## What is it?

Tree-based models can tell you **which features they used the most** to make decisions.

For the Iris dataset, which measurements matter most to identify the species?
- Petal length? Petal width? Sepal length?

## How it's calculated (Gini Importance):

Every time the tree splits on a feature, it reduces the **Gini impurity** (makes nodes purer).

Feature importance = total reduction in impurity caused by that feature across ALL trees.

All importances sum to 1.0 (they're proportions).

## Why does this matter?
- You can **remove unimportant features** → simpler, faster models
- You **understand what drives predictions** → trust the model
- You can find **data collection priorities** → collect what matters

In [ ]:
# ── Feature Importance from all models ───────────────────────────
fi_dt  = pd.Series(dt.feature_importances_,         index=iris.feature_names, name='Decision Tree')
fi_rf  = pd.Series(best_rf.feature_importances_,    index=iris.feature_names, name='Random Forest')
fi_xgb = pd.Series(xgb_model.feature_importances_,  index=iris.feature_names, name='XGBoost')

fi_df = pd.DataFrame([fi_dt, fi_rf, fi_xgb]).T

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'forestgreen', 'darkorange']

for ax, col, color in zip(axes, fi_df.columns, colors):
    sorted_fi = fi_df[col].sort_values(ascending=True)
    bars = ax.barh(sorted_fi.index, sorted_fi.values, color=color, edgecolor='white', height=0.6)
    ax.set_xlim(0, 1.0)
    ax.set_title(f'Feature Importance\n{col}', fontsize=12)
    ax.set_xlabel('Importance Score')
    # Add value labels
    for bar, val in zip(bars, sorted_fi.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Feature Importance — Which Measurements Matter Most?', fontsize=14)
plt.tight_layout()
plt.show()

print('💡 Key Insight:')
print('   Petal measurements dominate! This makes biological sense:')
print('   Petals vary dramatically between Iris species.')
print('   Sepal measurements overlap more across species.')

---
# 8. 📏 Evaluation Metrics — How Good Is Your Model, Really?

## Why accuracy alone is not enough

Imagine a cancer screening model. In 100 patients, 5 have cancer.
A **lazy model that always says "no cancer"** gets 95% accuracy!
But it misses ALL cancer patients — terrible in practice.

We need metrics that capture different aspects of performance.

## The Confusion Matrix (for 2 classes)

```
                    PREDICTED
                  Positive  Negative
ACTUAL  Positive  [  TP  ]  [  FN  ]
        Negative  [  FP  ]  [  TN  ]
```

| Term | What it means | Example |
|------|---------------|--------|
| **TP** True Positive | Said YES, was YES | Correctly identified cancer |
| **TN** True Negative | Said NO, was NO  | Correctly cleared patient |
| **FP** False Positive (Type I Error) | Said YES, was NO | False alarm |
| **FN** False Negative (Type II Error) | Said NO, was YES | **Missed disease!** |

## Key Metrics

$$\text{Accuracy} = \frac{TP + TN}{\text{Total}} \quad \text{(overall correctness)}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(when I say positive, how often am I right?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all actual positives, how many did I catch?)}$$

$$\text{F1 Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(balance of both)}$$

## When to use what:
| Situation | Use | Why |
|-----------|-----|-----|
| Medical diagnosis | **Recall** | Missing disease is catastrophic |
| Spam filter | **Precision** | False alarms (blocking real email) is bad |
| Balanced data | **Accuracy** or **F1** | Both classes matter equally |
| Comparing models | **ROC-AUC** | Threshold-independent comparison |

In [ ]:
# ── Confusion Matrices for All Models ────────────────────────────
models = {
    'Logistic Regression': (log_reg, log_reg.predict(X_i_test_sc)),
    'Decision Tree':       (dt, dt_preds),
    'Random Forest':       (best_rf, best_rf.predict(X_i_test)),
    'XGBoost':             (xgb_model, xgb_preds)
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
class_names = ['Setosa', 'Versicolor', 'Virginica']

for ax, (name, (model, preds)) in zip(axes, models.items()):
    cm = confusion_matrix(y_i_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_i_test, preds)
    ax.set_title(f'{name}\nAcc={acc:.3f}', fontsize=11)

plt.suptitle('Confusion Matrices — All Models\n'
             'Diagonal = correct predictions | Off-diagonal = mistakes', fontsize=13)
plt.tight_layout()
plt.show()

print('How to read: Row = actual class, Column = predicted class')
print('Perfect model → all numbers on the diagonal (top-left to bottom-right)')

In [ ]:
# ── Compare All Metrics Side by Side ─────────────────────────────
metrics_rows = []

for name, (model, preds) in models.items():
    # For multiclass ROC-AUC, use 'ovr' (one-vs-rest) strategy
    if hasattr(model, 'predict_proba'):
        if name == 'Logistic Regression':
            proba = model.predict_proba(X_i_test_sc)
        else:
            proba = model.predict_proba(X_i_test)
        auc = roc_auc_score(y_i_test, proba, multi_class='ovr')
    else:
        auc = None
    
    metrics_rows.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_i_test, preds),
        'Precision': precision_score(y_i_test, preds, average='weighted'),
        'Recall':    recall_score(y_i_test, preds, average='weighted'),
        'F1 Score':  f1_score(y_i_test, preds, average='weighted'),
        'ROC-AUC':   auc
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Model').round(4)
print('📊 Model Comparison Table:')
print(metrics_df.to_string())

# Highlight the best value in each column
print('\n💡 Note: All metrics range from 0 to 1. Higher = better.')
print('   For this simple dataset, all models perform very well!')

In [ ]:
# ── Visual Comparison ─────────────────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
plot_df = metrics_df[metrics_to_plot]

ax = plot_df.plot(kind='bar', figsize=(12, 5), edgecolor='white', width=0.75)
plt.title('Model Comparison: All Evaluation Metrics', fontsize=14)
plt.xlabel('Model')
plt.ylabel('Score')
plt.ylim(0.85, 1.05)
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Regression Metrics (for Linear Regression) ───────────────────
print('📊 REGRESSION METRICS — Quick Reference')
print('='*55)

mae  = mean_absolute_error(y_h_test, y_h_pred)
mse  = mean_squared_error(y_h_test, y_h_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_h_test, y_h_pred)

print(f"""
MAE  = {mae:.4f}
  Full name: Mean Absolute Error
  Formula:   average of |actual - predicted|
  Meaning:   On average, predictions are off by {mae:.3f} units (×$100k = ${mae*100:.0f}k)
  When good: Easy to interpret, same units as target

MSE  = {mse:.4f}
  Full name: Mean Squared Error
  Formula:   average of (actual - predicted)²
  Meaning:   Penalizes LARGE errors more than small ones
  When good: When big mistakes are especially bad

RMSE = {rmse:.4f}
  Full name: Root Mean Squared Error
  Formula:   √MSE
  Meaning:   Same units as target, but punishes outliers
  Note:      Always ≥ MAE

R²   = {r2:.4f}
  Full name: R-squared (Coefficient of Determination)
  Formula:   1 - (SS_residuals / SS_total)
  Meaning:   Model explains {r2*100:.1f}% of the variation in house prices
  Range:     0 to 1 (higher is better; can go negative if very bad!)
""")

---
# 9. 🔄 Cross-Validation — More Reliable Testing

## The Problem with a Single Train/Test Split

When you split data once (e.g., 80% train / 20% test), your results depend on **which 20% ended up in the test set** by luck.

Imagine you happened to get all the easy flowers in the test set → great score!
Or all the hard ones → terrible score!

## Cross-Validation Solution: Test on Multiple Splits

**5-Fold Cross Validation:**
```
Fold 1:  [TEST ][──────── TRAIN ────────]  → Score 1
Fold 2:  [TRAIN][TEST ][───── TRAIN ────]  → Score 2
Fold 3:  [TRAIN][──TRAIN──][TEST][─ TRN]  → Score 3
Fold 4:  [TRAIN][──────TRAIN────][TEST ]  → Score 4
Fold 5:  [─────────── TRAIN ────][TEST ]  → Score 5

Final Score = Mean(Score1...Score5) ± Std
```

Every sample gets tested exactly once. Results are much more reliable!

## Stratified K-Fold
For classification, use **Stratified** K-Fold: each fold has the same class proportions as the full dataset.

Example: If 33% are Setosa overall, each fold will have ~33% Setosa.

In [ ]:
# ── 5-Fold Stratified Cross-Validation ───────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, learning_rate=0.1,
                                              random_state=42, verbosity=0)
}

cv_results = {}
print('Cross-Validation Results (5-Fold, scoring=accuracy):')
print('='*70)
for name, model in cv_models.items():
    # Use scaled data for Logistic Regression, unscaled for trees
    X_cv = X_i_train_sc if 'Logistic' in name else X_i_train
    y_cv = y_i_train
    scores = cross_val_score(model, X_cv, y_cv, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s} Scores: {scores.round(3)}  Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')

print()
print('💡 Low Std = model gives consistent results regardless of which data you test on')
print('   High Std = results vary a lot = less reliable (maybe overfit to specific splits)')

In [ ]:
# ── Visualize CV Score Distribution ──────────────────────────────
cv_df = pd.DataFrame(cv_results)

fig, ax = plt.subplots(figsize=(11, 5))
bp = cv_df.boxplot(ax=ax, patch_artist=True,
              boxprops=dict(facecolor='lightblue', color='steelblue'),
              medianprops=dict(color='red', linewidth=2),
              whiskerprops=dict(color='steelblue'),
              capprops=dict(color='steelblue'))

ax.set_title('5-Fold CV Accuracy Distribution per Model', fontsize=13)
ax.set_ylabel('Accuracy Score')
ax.set_ylim(0.8, 1.05)

# Add mean labels
for i, col in enumerate(cv_df.columns, 1):
    mean_val = cv_df[col].mean()
    ax.text(i, mean_val + 0.008, f'μ={mean_val:.3f}',
            ha='center', fontsize=10, color='darkred', fontweight='bold')

plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

print('Reading the boxplot:')
print('  Box = middle 50% of scores (25th to 75th percentile)')
print('  Red line = median score across folds')
print('  Whiskers = min and max scores')
print('  Smaller box = more consistent = more trustworthy model')

---
# 10. 📋 Model Card — Document Your Model!

## What is a Model Card?

A **Model Card** is a short document that explains your ML model to others (and your future self!).

Think of it like a **nutrition label on food** — it tells you exactly what's inside, any warnings, and who it's for.

First introduced by Google researchers (Mitchell et al., 2019), now standard in industry.

## Why write one?
- **Transparency**: Others can understand what your model does and doesn't do
- **Trust**: Stakeholders can see performance evidence
- **Safety**: Documents limitations and risks before deployment
- **Fairness**: Forces you to think about who the model might harm

In [ ]:
# ── Model Card: XGBoost Iris Classifier ───────────────────────────

xgb_final_acc = accuracy_score(y_i_test, xgb_preds)
xgb_final_f1  = f1_score(y_i_test, xgb_preds, average='weighted')
xgb_cv_scores = cv_results['XGBoost']

model_card = f"""
╔══════════════════════════════════════════════════════════════════════╗
║            MODEL CARD — Iris Species Classifier (XGBoost)           ║
╚══════════════════════════════════════════════════════════════════════╝

1. MODEL DETAILS
   ├── Name         : Iris Species Classifier v1.0
   ├── Type         : XGBoost Multi-class Classification
   ├── Output       : One of [Setosa, Versicolor, Virginica]
   ├── Date Trained : 2024
   └── Framework    : XGBoost + scikit-learn

2. INTENDED USE
   ├── Primary Use  : Classify Iris flowers by species from measurements
   ├── Audience     : ML students learning classification
   └── Out of Scope : Real-time botanical identification for conservation

3. TRAINING DATA
   ├── Source       : UCI Iris Dataset (built into sklearn)
   ├── Size         : 150 samples total
   ├── Features     : sepal length, sepal width, petal length, petal width
   ├── Classes      : Setosa (50), Versicolor (50), Virginica (50) — perfectly balanced
   ├── Split        : 80% train (120 samples) / 20% test (30 samples)
   └── Preprocessing: None needed (no missing values, no encoding)

4. PERFORMANCE (Test Set, 30 samples)
   ├── Accuracy     : {xgb_final_acc:.4f} ({xgb_final_acc*100:.1f}%)
   ├── F1 Score     : {xgb_final_f1:.4f} (weighted average across 3 classes)
   ├── CV Mean Acc  : {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}
   └── Note         : Setosa is always predicted perfectly; occasional confusion
                      between Versicolor and Virginica (similar petal sizes)

5. LIMITATIONS
   ├── Very small dataset (150 samples) — may not generalise to wild populations
   ├── Collected in 1936 — genetic diversity may have changed since
   ├── Only 4 features — real identification might need more data
   └── Perfect class balance — real-world may have unequal species frequency

6. KEY HYPERPARAMETERS
   ├── n_estimators     : 100
   ├── max_depth        : 4
   ├── learning_rate    : 0.1
   ├── subsample        : 0.8
   └── colsample_bytree : 0.8

7. ETHICAL CONSIDERATIONS
   └── Low risk: classifying flowers has no direct societal harm.
       However, in real deployments, model cards for high-stakes systems
       must include bias analysis, fairness metrics, and known failure modes.
"""

print(model_card)

---
# 🎓 Summary — Everything You Learned

## The 5 Models

| Model | Type | Key Idea | Best When |
|-------|------|----------|-----------|
| **Linear Regression** | Regression | Fit a line to predict numbers | Continuous output, linear relationships |
| **Logistic Regression** | Classification | Sigmoid squeezes output to 0-1 probability | Binary/multiclass, need interpretability |
| **Decision Tree** | Both | Flowchart of yes/no questions | Need explainability, quick prototype |
| **Random Forest** | Both | 100s of trees voting together (Bagging) | Noisy data, need robustness |
| **XGBoost** | Both | Trees fixing each other's mistakes (Boosting) | Best accuracy, Kaggle competitions |

## The Key Concepts

| Concept | One-liner |
|---------|----------|
| **Overfitting** | Model memorizes training data, fails on new data |
| **Underfitting** | Model too simple to capture patterns |
| **Feature Scaling** | Bring all features to same scale (for linear models) |
| **Train/Test Split** | Always hold out data to test how model does on NEW data |
| **Cross-Validation** | Multiple splits = more reliable performance estimate |
| **Hyperparameter Tuning** | Search for best model settings using CV |
| **Feature Importance** | Which inputs does the model rely on most? |

## The Metrics

| Metric | For | Use When |
|--------|-----|----------|
| **MAE / RMSE / R²** | Regression | Predicting numbers |
| **Accuracy** | Classification | Balanced classes, general overview |
| **Precision** | Classification | False positives are costly (spam filter) |
| **Recall** | Classification | False negatives are costly (medical!) |
| **F1 Score** | Classification | Imbalanced classes |
| **ROC-AUC** | Classification | Compare models across all thresholds |

---

## 🚀 What to Learn Next
1. **Support Vector Machines (SVM)** — finds the widest possible margin between classes
2. **K-Nearest Neighbors (KNN)** — classify based on what your nearest neighbors are
3. **Neural Networks** — the backbone of deep learning and AI
4. **Feature Engineering** — creating better features from raw data
5. **Handling Imbalanced Data** — SMOTE, class weights, undersampling

> *"All models are wrong, but some are useful."* — George Box  
> Start simple. Measure everything. Always question your assumptions!